In [3]:
import import_ipynb
from load_process_data import load_data, preprocess_data, truncate_pad_data, shuffle_data
from models import BaseGRU, MultiHeadCnnRnn
from keras.regularizers import l2
from model_eval import cross_validate
from visualize import visualize_accuracies
import os


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.6 MB/s eta 0:00:00


In [2]:
# Define hyperparameters

input_shape = (81, 23)  # 81 timesteps with 23 features
num_folds = 5  # Cross-validation
epochs = 30
percentage_map = {
    (50, 75): 1  # 25% of data with 75-100% of the original length
}

test_splits = {(49, 50): 1}

# GRU hyperparameters

learning_rate = 0.002
dropout = 0.1
recurrent_dropout = 0.1
kernel_regularizer = l2(0.01)  # Regularization strength for kernel
recurrent_regularizer = l2(0.01)  # Regularization strength for recurrent connections

# CNN hyperparameters

kernel_sizes = [8, 5, 3]
filters = [16, 32, 64]
learning_rate = 0.001
weight_decay = 0.01

NameError: name 'l2' is not defined

In [ ]:
# Create models with regularization

GRUModel = BaseGRU(input_shape=input_shape,
                learning_rate=learning_rate,
                dropout=dropout,
                recurrent_dropout=recurrent_dropout,
                kernel_regularizer=kernel_regularizer,
                recurrent_regularizer=recurrent_regularizer)

MultiHeadModel = MultiHeadCnnRnn(input_shape=input_shape,
                                 kernel_sizes=kernel_sizes,
                                 filters=filters,
                                 learning_rate=learning_rate,
                                 weight_decay=weight_decay,
                                 dropout=dropout,
                                 recurrent_dropout=recurrent_dropout,
                                 kernel_regularizer=kernel_regularizer,
                                 recurrent_regularizer=recurrent_regularizer)

Feature amount: 23


In [ ]:
# Loading data

current_directory = os.getcwd()
parent_directory = os.path.dirname(current_directory)
data_path = os.path.join(parent_directory, 'data/')
train_y_list, test_y_list, train_X_list, test_X_list = load_data(data_path)

# Preprocessing data (reshape + standardization)

train_x_list_filtered, test_x_list_filtered = preprocess_data(train_X_list, test_X_list)
print(train_x_list_filtered[0].shape)

Loading data ...
Data shapes: 
train_x_list: (5,)
training data within each fold: (16063, 81, 25)
train_y_list: (5,)
training label within each fold: (16063,)
Preprocessing data ...
(16063, 81, 23)


In [ ]:
# Reshaping time series train / test data shape for real-time forecasting

train_x_list_random = []
train_y_list_random = []
test_x_list_random = []
test_y_list_random = []

for i, fold in enumerate(train_x_list_filtered):
    train_x_list_truncated = truncate_pad_data(fold, percentage_map, input_shape[0])
    train_x_list_shuffled, train_y_list_shuffled = shuffle_data(train_x_list_truncated, train_y_list[i])
    print(train_x_list_shuffled)
    train_x_list_random.append(train_x_list_shuffled)
    train_y_list_random.append(train_y_list_shuffled)

for i, fold in enumerate(test_x_list_filtered):
    test_x_list_truncated = truncate_pad_data(fold, test_splits, input_shape[0])
    test_x_list_shuffled, test_y_list_shuffled = shuffle_data(test_x_list_truncated, test_y_list[i])
    test_x_list_random.append(test_x_list_shuffled)
    test_y_list_random.append(test_y_list_shuffled)

print(f"Fold shape: {train_x_list_random[0].shape}")
print(f"Fold shape: {train_y_list_random[0].shape}")
print(f"Fold shape: {test_x_list_random[0].shape}")
print(f"Fold shape: {test_y_list_random[0].shape}")


[[[ 0.45966548  0.8496059   1.1224996  ...  0.36933222 -0.4602203
   -3.826269  ]
  [ 0.45924735  0.8488447   1.1224996  ...  0.38108996 -0.4602203
   -3.826269  ]
  [ 0.45882523  0.8480856   1.1224996  ...  0.3905098  -0.4602203
   -3.826269  ]
  ...
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]]

 [[ 2.2637842  -2.308251    1.12896    ...  0.2865788  -0.17410941
    0.26715782]
  [ 2.2641144  -2.309166    1.12896    ...  0.2734968  -0.17410941
    0.26715782]
  [ 2.2644157  -2.3100932   1.12896    ...  0.26998988 -0.17410941
    0.26715782]
  ...
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]]

 [[ 1.4329808  -1.1657451   1.1224996  

In [ ]:
print(f"Fold shape: {train_x_list_random[0].shape}")
print(f"Fold shape: {train_y_list_random[0].shape}")
print(f"Fold shape: {test_x_list_random[0].shape}")
print(f"Fold shape: {test_y_list_random[0].shape}")

Fold shape: (16062, 81, 23)
Fold shape: (16062,)
Fold shape: (4016, 81, 23)
Fold shape: (4016,)


In [ ]:
train_accuracies, test_accuracies = cross_validate(GRUModel, num_folds, train_x_list_filtered,
                                                   train_y_list, test_x_list_filtered, test_y_list)

Testing on Base GRU class: 
Training on fold 1/5
Epoch 1/10
502/502 [==============================] - 19s 39ms/step - loss: 0.5172 - accuracy: 0.7909 - val_loss: 0.4582 - val_accuracy: 0.8222
Epoch 2/10
324/502 [==================>...........] - ETA: 6s - loss: 0.4786 - accuracy: 0.8184

KeyboardInterrupt: 

In [ ]:
# BaseGRU training and testing
print(train_x_list_random[4].shape)
print(train_y_list_random[4].shape)
print(test_x_list_filtered[4].shape)
print(test_y_list[4].shape)

truncated_train_accuracies, truncated_test_accuracies = cross_validate(GRUModel, num_folds, train_x_list_random,
                                                   train_y_list_random, test_x_list_filtered, test_y_list)


(16064, 81, 23)
(16064,)
(4015, 81, 23)
(4015,)
Testing on Base GRU class: 
Training on fold 1/5
Epoch 1/10
502/502 [==============================] - 22s 40ms/step - loss: 0.8339 - accuracy: 0.6260 - val_loss: 0.6598 - val_accuracy: 0.6683
Epoch 2/10
502/502 [==============================] - 19s 39ms/step - loss: 0.6670 - accuracy: 0.6566 - val_loss: 0.6257 - val_accuracy: 0.7092
Epoch 3/10
502/502 [==============================] - 19s 38ms/step - loss: 0.6592 - accuracy: 0.6644 - val_loss: 0.6229 - val_accuracy: 0.7199
Epoch 4/10
502/502 [==============================] - 19s 38ms/step - loss: 0.6559 - accuracy: 0.6681 - val_loss: 0.6257 - val_accuracy: 0.7007
Epoch 5/10
502/502 [==============================] - 19s 38ms/step - loss: 0.6549 - accuracy: 0.6652 - val_loss: 0.6161 - val_accuracy: 0.7273
Epoch 6/10
502/502 [==============================] - 19s 38ms/step - loss: 0.6533 - accuracy: 0.6669 - val_loss: 0.6222 - val_accuracy: 0.7276
Epoch 7/10
502/502 [===================

KeyboardInterrupt: 